<a href="https://colab.research.google.com/github/sAI-2025/Intelligent-Customer-Signal-Detector/blob/main/telco_churn_llm_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Churn — Feature Remap, Sampling & LLM-Generated Support Chats (Ollama on Colab)

**Pipeline overview**
1. Load `Telco_customer_churn.csv`
2. Rename columns to the target schema (per the reference tree)
3. Stratified sample: 1000 churned + 1000 not-churned (2000 total)
4. Drop columns not present in the reference tree
5. Add derived fields: `satisfaction_level`, `service_count`, `support_interaction_count`, `open_issue_count`, `close_issue_count`, `resolution_status_open_closed`
6. Install & run **Ollama** in Colab, pull a 3B model (`qwen2.5:3b`, swap for `gemma2:2b` if you prefer)
7. Use the LLM to generate a **support chat transcript** per customer (saved as `<customer_id>.txt`), then a **feedback_text**, then classify **feedback_sentiment** (Positive/Negative/Neutral)
8. Export enriched CSV + zipped chat transcripts

> ⚠️ Runtime note: with ~2000 customers × 3 LLM calls each = ~6000 generations, a 3B model on a free T4 GPU can take a long while. Start with `TEST_MODE=True` (10 rows) to validate the whole pipeline before running the full 2000.


## 1. Setup — packages

In [1]:
!pip install -q requests tqdm pandas numpy


## 2. Install Ollama and start the server in the background
This runs `ollama serve` as a background subprocess so the notebook cell doesn't block.


In [2]:
!apt-get update -qq
!apt-get install -y zstd


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 67 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (442 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, os, time

def start_ollama_server():
    proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=open("ollama.log", "w"),
        stderr=open("ollama_error.log", "w"),
        preexec_fn=os.setsid,
    )
    time.sleep(5)
    print("Ollama server started, PID:", proc.pid)
    return proc

ollama_proc = start_ollama_server()


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama server started, PID: 2953


In [4]:
!which ollama
!ollama --version


/usr/local/bin/ollama
ollama version is 0.32.9


## 3. Pull a small open-source model
`qwen2.5:3b` (~2GB) is a good balance of quality/speed on free Colab GPUs.
Swap to `gemma2:2b` if you want something even lighter.


In [18]:
MODEL_NAME = "gemma2:2b" #"qwen2.5:3b"   # or "gemma2:2b"
!ollama pull {MODEL_NAME}


## 4. Sanity-check the model


In [6]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())


CUDA available: True
GPU: Tesla T4
GPU count: 1


In [19]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"

def ollama_generate(prompt, model=MODEL_NAME, temperature=0.7, max_tokens=300, retries=2):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": max_tokens},
    }
    for attempt in range(retries + 1):
        try:
            r = requests.post(OLLAMA_URL, json=payload, timeout=120)
            r.raise_for_status()
            return r.json().get("response", "").strip()
        except Exception as e:
            if attempt == retries:
                return f"[ERROR: {e}]"
            time.sleep(2)

print(ollama_generate("Say hello in one short sentence."))


Hello! 😊


In [20]:
!ollama ps


NAME         ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
gemma2:2b    8ccf136fdd52    1.9 GB    100% GPU     4096       4 minutes from now    


## 5. Load the dataset
Upload `Telco_customer_churn.csv` to the Colab file browser (left sidebar) first,
or mount Google Drive and point `CSV_PATH` at the file there.


In [21]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

CSV_PATH = "/content/drive/MyDrive/FirstSource/Telco_customer_churn.csv"

df_raw = pd.read_csv(
    CSV_PATH,
    skipinitialspace=True
)


In [56]:
with open(CSV_PATH, "r", encoding="utf-8") as f:
    for i in range(3):
        print(f"LINE {i}:")
        print(repr(f.readline()))



LINE 0:
'customerID, gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines   , InternetService, OnlineSecurity     , OnlineBackup       , DeviceProtection   , TechSupport        , StreamingTV        , StreamingMovies    , Contract      , PaperlessBilling, PaymentMethod            , MonthlyCharges, TotalCharges, Churn, churn_rate, Churn Score, CLTV, Churn Reason                             , Count, Country      , State     , City                  , Zip Code, Lat Long                , Latitude , Longitude  , Age, Under 30, Married, Referred a Friend, Number of Referrals, Offer  , Avg Monthly Long Distance Charges, Avg Monthly GB Download, Streaming Music, Premium Tech Support, Unlimited Data, Total Refunds, Total Extra Data Charges, Total Long Distance Charges, Total Revenue        , Satisfaction Score, Customer Status, Churn Score.1, Churn Category\n'
LINE 1:
'3668-QPYBK, Male  ,             0, No     , No        ,      2, Yes         , No              , DSL   

In [57]:
print(df_raw.iloc[0].to_dict())


{'customerID': '3668-QPYBK', 'gender': 'Male  ', 'SeniorCitizen': 0, 'Partner': 'No     ', 'Dependents': 'No        ', 'tenure': 2, 'PhoneService': 'Yes         ', 'MultipleLines   ': 'No              ', 'InternetService': 'DSL            ', 'OnlineSecurity     ': 'Yes                ', 'OnlineBackup       ': 'Yes                ', 'DeviceProtection   ': 'No                 ', 'TechSupport        ': 'No                 ', 'StreamingTV        ': 'No                 ', 'StreamingMovies    ': 'No                 ', 'Contract      ': 'Month-to-month', 'PaperlessBilling': 'Yes             ', 'PaymentMethod            ': 'Mailed check             ', 'MonthlyCharges': 53.85, 'TotalCharges': 108.15, 'Churn': 'Yes  ', 'churn_rate': 1, 'Churn Score': 86, 'CLTV': 3239, 'Churn Reason                             ': 'Competitor made better offer             ', 'Count': 1, 'Country      ': 'United States', 'State     ': 'California', 'City                  ': 'Los Angeles           ', 'Zip Code': 900

In [58]:
df_raw.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines   ',
 'InternetService',
 'OnlineSecurity     ',
 'OnlineBackup       ',
 'DeviceProtection   ',
 'TechSupport        ',
 'StreamingTV        ',
 'StreamingMovies    ',
 'Contract      ',
 'PaperlessBilling',
 'PaymentMethod            ',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'churn_rate',
 'Churn Score',
 'CLTV',
 'Churn Reason                             ',
 'Count',
 'Country      ',
 'State     ',
 'City                  ',
 'Zip Code',
 'Lat Long                ',
 'Latitude ',
 'Longitude  ',
 'Age',
 'Under 30',
 'Married',
 'Referred a Friend',
 'Number of Referrals',
 'Offer  ',
 'Avg Monthly Long Distance Charges',
 'Avg Monthly GB Download',
 'Streaming Music',
 'Premium Tech Support',
 'Unlimited Data',
 'Total Refunds',
 'Total Extra Data Charges',
 'Total Long Distance Charges',
 'Total Revenue        ',
 'Satisfaction Score',
 'Customer Status',
 'C

In [59]:
df_raw.rename(columns={
    'MultipleLines   ': 'MultipleLines',
    'OnlineSecurity     ': 'OnlineSecurity',
    'OnlineBackup       ': 'OnlineBackup',
    'DeviceProtection   ': 'DeviceProtection',
    'TechSupport        ': 'TechSupport',
    'StreamingTV        ': 'StreamingTV',
    'StreamingMovies    ': 'StreamingMovies',
    'Contract      ': 'Contract',
    'PaymentMethod            ': 'PaymentMethod',
    'Churn Reason                             ': 'Churn Reason',
    'Country      ': 'Country',
    'State     ': 'State',
    'City                  ': 'City',
    'Lat Long                ': 'Lat Long',
    'Latitude ': 'Latitude',
    'Longitude  ': 'Longitude',
    'Offer  ': 'Offer',
    'Total Revenue        ': 'Total Revenue'
}, inplace=True)


In [60]:
df_raw.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'churn_rate',
 'Churn Score',
 'CLTV',
 'Churn Reason',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Age',
 'Under 30',
 'Married',
 'Referred a Friend',
 'Number of Referrals',
 'Offer',
 'Avg Monthly Long Distance Charges',
 'Avg Monthly GB Download',
 'Streaming Music',
 'Premium Tech Support',
 'Unlimited Data',
 'Total Refunds',
 'Total Extra Data Charges',
 'Total Long Distance Charges',
 'Total Revenue',
 'Satisfaction Score',
 'Customer Status',
 'Churn Score.1',
 'Churn Category']

In [61]:
print(df_raw.shape)
print(df_raw['Churn'].value_counts())


(7043, 52)
Churn
No       5174
Yes      1869
Name: count, dtype: int64


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 6. Stratified sampling — 1000 Churned + 1000 Not-Churned
Sampling is done on the **original** `Churn` (Yes/No) column, before it gets dropped later.


In [65]:
print(df_raw['Churn'].value_counts())


Churn
No       5174
Yes      1869
Name: count, dtype: int64


In [66]:
print(df_raw['Churn'].unique())


['Yes  ' 'No   ']


In [67]:
df_raw['Churn'] = df_raw['Churn'].replace(
    r'^\s+|\s+$',
    '',
    regex=True
)


In [69]:
print(df_raw['Churn'].unique())
print(df_raw['Churn'].value_counts())


['Yes' 'No']
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [70]:
N_PER_CLASS = 1000

churned = df_raw[df_raw['Churn'] == 'Yes'].sample(n=N_PER_CLASS, random_state=42)
not_churned = df_raw[df_raw['Churn'] == 'No'].sample(n=N_PER_CLASS, random_state=42)

df_sample = pd.concat([churned, not_churned], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
print(df_sample.shape)
print(df_sample['Churn'].value_counts())

(2000, 52)
Churn
No     1000
Yes    1000
Name: count, dtype: int64


In [71]:
df_sample.shape

(2000, 52)

In [72]:
df_sample.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Premium Tech Support,Unlimited Data,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Score.1,Churn Category
0,4795-UXVCJ,Male,0,No,No,26,Yes,No,No,No internet service,...,No,No,0.0,0,1188.20,1645.50,3,Stayed,77,0
1,1915-IOFGU,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,Yes,0.0,0,41.39,111.89,2,Churned,90,Competitor
2,9680-NIAUV,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,No,Yes,0.0,0,1343.52,9472.82,4,Stayed,74,0


In [74]:
df_sample.to_csv("/content/drive/MyDrive/FirstSource/Telco_customer_churn_sample.csv", index=False)


## 7. Rename columns to the target schema
Mapping follows the reference feature tree exactly. Columns not mentioned anywhere in the
tree (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `Premium Tech Support`,
`CLTV`, `Churn Reason`, `Count`, `Customer Status`, `Churn Score.1`, `Churn Category`,
`Avg Monthly Long Distance Charges`, and the raw `Churn`/`Satisfaction Score` columns that get
replaced by derived fields) are dropped in the next step.


In [75]:
df_sample = pd.read_csv("/content/drive/MyDrive/FirstSource/Telco_customer_churn_sample.csv",low_memory=False)


In [76]:
df_sample.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'churn_rate',
 'Churn Score',
 'CLTV',
 'Churn Reason',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Age',
 'Under 30',
 'Married',
 'Referred a Friend',
 'Number of Referrals',
 'Offer',
 'Avg Monthly Long Distance Charges',
 'Avg Monthly GB Download',
 'Streaming Music',
 'Premium Tech Support',
 'Unlimited Data',
 'Total Refunds',
 'Total Extra Data Charges',
 'Total Long Distance Charges',
 'Total Revenue',
 'Satisfaction Score',
 'Customer Status',
 'Churn Score.1',
 'Churn Category']

In [77]:
rename_map = {
    'customerID': 'customer_id',
    'gender': 'gender',
    'SeniorCitizen': 'senior_citizen',
    'Partner': 'partner',
    'Dependents': 'dependents',
    'tenure': 'tenure_months',
    'PhoneService': 'phone_service',
    'MultipleLines': 'multiple_lines',
    'InternetService': 'internet_service',
    'OnlineSecurity': 'online_security',
    'OnlineBackup': 'online_backup',
    'DeviceProtection': 'device_protection',
    'TechSupport': 'tech_support',
    'StreamingTV': 'streaming_tv',
    'StreamingMovies': 'streaming_movies',
    'Contract': 'contract_type',
    'PaperlessBilling': 'paperless_billing',
    'PaymentMethod': 'payment_method',
    'MonthlyCharges': 'monthly_charges',
    'TotalCharges': 'total_charges',
    'Churn': 'churn',
    'churn_rate': 'churn_rate',
    'Churn Score': 'churn_score_target',
    'CLTV': 'cltv',
    'Churn Reason': 'churn_reason',
    'Count': 'count',
    'Country': 'country',
    'State': 'state',
    'City': 'city',
    'Zip Code': 'zip_code',
    'Lat Long': 'lat_long',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Age': 'age',
    'Under 30': 'under_30',
    'Married': 'married',
    'Referred a Friend': 'referred_a_friend',
    'Number of Referrals': 'number_of_referrals',
    'Offer': 'offer',
    'Avg Monthly Long Distance Charges': 'avg_monthly_long_distance_charges',
    'Avg Monthly GB Download': 'avg_monthly_gb_download',
    'Streaming Music': 'streaming_music',
    'Premium Tech Support': 'premium_tech_support',
    'Unlimited Data': 'unlimited_data',
    'Total Refunds': 'total_refunds',
    'Total Extra Data Charges': 'total_extra_data_charges',
    'Total Long Distance Charges': 'total_long_distance_charges',
    'Total Revenue': 'total_revenue',
    'Satisfaction Score': 'satisfaction_score_raw',
    'Customer Status': 'customer_status',
    'Churn Score.1': 'churn_score_1',
    'Churn Category': 'churn_category'
}

df = df_sample.rename(columns=rename_map)

print(df.columns.tolist())


['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract_type', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn', 'churn_rate', 'churn_score_target', 'cltv', 'churn_reason', 'count', 'country', 'state', 'city', 'zip_code', 'lat_long', 'latitude', 'longitude', 'age', 'under_30', 'married', 'referred_a_friend', 'number_of_referrals', 'offer', 'avg_monthly_long_distance_charges', 'avg_monthly_gb_download', 'streaming_music', 'premium_tech_support', 'unlimited_data', 'total_refunds', 'total_extra_data_charges', 'total_long_distance_charges', 'total_revenue', 'satisfaction_score_raw', 'customer_status', 'churn_score_1', 'churn_category']


In [79]:
df.head(4)

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,...,premium_tech_support,unlimited_data,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,satisfaction_score_raw,customer_status,churn_score_1,churn_category
0,4795-UXVCJ,Male,0,No,No,26,Yes,No,No,No internet service,...,No,No,0.00,0,1188.20,1645.50,3,Stayed,77,0
1,1915-IOFGU,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,Yes,0.00,0,41.39,111.89,2,Churned,90,Competitor
2,9680-NIAUV,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,No,Yes,0.00,0,1343.52,9472.82,4,Stayed,74,0
3,5519-NPHVG,Female,0,No,No,12,Yes,Yes,Fiber optic,No,...,No,No,7.39,120,153.00,1311.71,3,Churned,73,Price


In [80]:
# df.to_csv("/content/drive/MyDrive/FirstSource/Telco_customer_churn_sample_renamed.csv", index=False)

## 8. Drop columns not present in the reference feature tree


In [81]:
df.shape

(2000, 52)

In [17]:
drop_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'Premium Tech Support', 'CLTV', 'Churn Reason', 'Count',
    'Customer Status', 'Churn Score.1', 'Churn Category',
    'Avg Monthly Long Distance Charges', 'Churn', 'churn_rate',
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
print(df.shape)
print(df.columns.tolist())


(2000, 38)
['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'streaming_tv', 'streaming_movies', 'contract_type', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_score_target', 'country', 'state', 'city', 'zip_code', 'lat_long', 'latitude', 'longitude', 'age', 'under_30', 'married', 'referred_a_friend', 'number_of_referrals', 'offer', 'avg_monthly_gb_download', 'streaming_music', 'unlimited_data', 'total_refunds', 'total_extra_data_charges', 'total_long_distance_charges', 'total_revenue', 'satisfaction_score_raw']


## 9. Add derived fields

- `satisfaction_level` — bucketed from the raw Satisfaction Score (1-5)
- `service_count` — count of active services (phone, internet, streaming tv/movies/music, unlimited data)
- `open_issue_count` / `close_issue_count` / `support_interaction_count` — rule-based on `churn_score_target`
- `resolution_status_open_closed` — random, 85% Resolved / 15% NotResolved


In [89]:
df= pd.read_csv("/content/drive/MyDrive/FirstSource/Telco_customer_churn_sample_renamed.csv")

In [91]:
df.head(3)

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,...,premium_tech_support,unlimited_data,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,satisfaction_score_raw,customer_status,churn_score_1,churn_category
0,4795-UXVCJ,Male,0,No,No,26,Yes,No,No,No internet service,...,No,No,0.0,0,1188.20,1645.50,3,Stayed,77,0
1,1915-IOFGU,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,Yes,0.0,0,41.39,111.89,2,Churned,90,Competitor
2,9680-NIAUV,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,No,Yes,0.0,0,1343.52,9472.82,4,Stayed,74,0


In [92]:
def satisfaction_level(score):
    if pd.isna(score):
        return 'Unknown'
    if score <= 2:
        return 'Low'
    elif score == 3:
        return 'Medium'
    else:
        return 'High'

df['satisfaction_level'] = df['satisfaction_score_raw'].apply(satisfaction_level)
df = df.drop(columns=['satisfaction_score_raw'])


def service_count(row):
    count = 0
    if row.get('phone_service') == 'Yes':
        count += 1
    if str(row.get('internet_service')) not in ['No', 'nan', 'None']:
        count += 1
    for col in ['streaming_tv', 'streaming_movies', 'streaming_music', 'unlimited_data']:
        if row.get(col) == 'Yes':
            count += 1
    return count

df['service_count'] = df.apply(service_count, axis=1)


def support_counts(churn_score):
    cs = 0 if pd.isna(churn_score) else churn_score
    if cs < 60:
        close_issue_count = int(cs // 10) + 1
        open_issue_count = random.randint(0, 1)
    else:
        close_issue_count = random.randint(0, 1)
        open_issue_count = int(cs // 10)
    return open_issue_count, close_issue_count

support_results = df['churn_score_target'].apply(support_counts)
df['open_issue_count'] = support_results.apply(lambda t: t[0])
df['close_issue_count'] = support_results.apply(lambda t: t[1])
df['support_interaction_count'] = df['open_issue_count'] + df['close_issue_count']

df['resolution_status_open_closed'] = np.random.choice(
    ['Resolved', 'NotResolved'], size=len(df), p=[0.85, 0.15]
)

print(df[['churn_score_target', 'open_issue_count', 'close_issue_count',
          'support_interaction_count', 'resolution_status_open_closed',
          'satisfaction_level', 'service_count']].head(10))


   churn_score_target  open_issue_count  close_issue_count  \
0                  77                 7                  1   
1                  90                 9                  0   
2                  74                 7                  0   
3                  73                 7                  1   
4                  39                 0                  4   
5                  75                 7                  1   
6                  87                 8                  0   
7                  25                 1                  3   
8                  77                 7                  1   
9                  28                 0                  3   

   support_interaction_count resolution_status_open_closed satisfaction_level  \
0                          8                      Resolved             Medium   
1                          9                      Resolved                Low   
2                          7                   NotResolved               H

In [93]:
df.head(3)

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,...,total_revenue,customer_status,churn_score_1,churn_category,satisfaction_level,service_count,open_issue_count,close_issue_count,support_interaction_count,resolution_status_open_closed
0,4795-UXVCJ,Male,0,No,No,26,Yes,No,No,No internet service,...,1645.50,Stayed,77,0,Medium,1,7,1,8,Resolved
1,1915-IOFGU,Female,0,No,No,1,Yes,No,Fiber optic,No,...,111.89,Churned,90,Competitor,Low,1,9,0,9,Resolved
2,9680-NIAUV,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,9472.82,Stayed,74,0,High,1,7,0,7,NotResolved


In [94]:
df.to_csv("/content/drive/MyDrive/FirstSource/Telco_customer_final_sample.csv", index=False)

## 10. LLM prompt builders
Chat topic is picked from `customer_chat_topics`. The **last line** of every generated chat is
controlled by `resolution_status_open_closed`:
- `NotResolved` → agent's closing line: *"It will be resolved by the end of the day."*
- `Resolved` → agent's closing line: *"Thank you for contacting us."*


In [43]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/FirstSource/Telco_customer_final_sample.csv")

In [44]:
customer_chat_topics = [

    # =========================================================
    # INTERNET SERVICE
    # =========================================================

    "internet connection slow or unstable",
    "download speed lower than subscribed plan",
    "upload speed lower than expected",
    "frequent internet disconnections",
    "internet completely unavailable",
    "Wi-Fi signal weak in certain rooms",
    "high latency or ping during usage",
    "network becomes slow during peak hours",
    "modem or router unable to establish connection",
    "internet stops working after restart",
    "connection drops when multiple devices connect",
    "websites load slowly despite active connection",
    "new device cannot connect to Wi-Fi",
    "network equipment not responding",
    "internet service activation problem",
    "internet service interruption after payment",
    "inconsistent internet connection quality",
    "customer wants to upgrade internet speed",
    "customer wants to downgrade internet plan",
    "internet speed differs across devices",


    # =========================================================
    # PHONE SERVICE
    # =========================================================

    "phone service unavailable",
    "calls dropping unexpectedly",
    "poor call quality",
    "incoming calls not connecting",
    "outgoing calls failing",
    "calls failing from specific locations",
    "phone signal weak or unavailable",
    "phone network connection disappearing intermittently",
    "phone cannot connect to carrier network",
    "device unable to register on network",
    "calling service stopped after configuration change",
    "customer cannot receive verification calls",
    "phone service activated but not working",
    "customer wants to change phone plan",
    "unexpected phone service charges",


    # =========================================================
    # MULTIPLE LINES
    # =========================================================

    "additional phone line cannot be activated",
    "existing secondary line not connecting",
    "multiple lines affecting network quality",
    "one phone line has no service",
    "calls failing between connected lines",
    "additional line missing from account",
    "customer wants to add another line",
    "customer wants to remove an existing line",
    "multiple-line pricing is unclear",
    "family or shared line benefits not working",
    "line transfer or replacement problem",
    "secondary device cannot connect",
    "one line has significantly slower service",


    # =========================================================
    # STREAMING TV
    # =========================================================

    "TV streaming buffering frequently",
    "TV streaming quality automatically decreases",
    "TV channels not loading",
    "live TV freezes during playback",
    "TV application crashes",
    "certain TV channels unavailable",
    "TV audio and video out of sync",
    "streaming stops while changing channels",
    "TV service unavailable despite active subscription",
    "remote or TV device cannot connect to service",
    "customer cannot access subscribed TV channels",
    "poor TV picture quality",
    "TV streaming performance worsens during peak hours",
    "customer wants to upgrade TV package",
    "customer dissatisfied with available TV content",


    # =========================================================
    # STREAMING MOVIES
    # =========================================================

    "movies buffering during playback",
    "movie playback stops unexpectedly",
    "movie video quality becomes blurry",
    "movies take too long to start",
    "selected movie unavailable",
    "movie catalog does not match subscription",
    "movie audio and video synchronization problem",
    "movie application crashes",
    "movie playback fails on specific devices",
    "downloaded movie cannot play",
    "movie streaming consumes excessive data",
    "customer cannot access premium movie content",
    "customer dissatisfied with movie selection",
    "customer wants to change movie package",
    "repeated playback errors during movies",


    # =========================================================
    # STREAMING MUSIC
    # =========================================================

    "music playback buffering",
    "songs stop unexpectedly",
    "music streaming quality is poor",
    "songs take too long to load",
    "music application crashes",
    "specific songs unavailable",
    "playlist fails to load",
    "music stops when switching networks",
    "audio cuts out repeatedly",
    "downloaded music cannot play offline",
    "customer cannot access included music service",
    "music service activation problem",
    "customer dissatisfied with music catalog",
    "customer wants to upgrade music service",
    "music streaming uses unexpected data",


    # =========================================================
    # UNLIMITED DATA
    # =========================================================

    "unlimited data feature not working",
    "mobile data becomes unusually slow",
    "customer believes unlimited data is restricted",
    "high-speed data appears to be throttled",
    "mobile data disconnects frequently",
    "data unavailable despite active plan",
    "customer cannot use data-intensive applications",
    "video streaming becomes extremely slow",
    "download speed decreases significantly",
    "data service stops after plan change",
    "unlimited data feature missing from account",
    "customer wants to activate unlimited data",
    "customer wants to remove unlimited data",
    "customer questions unlimited data pricing",
    "unexpected data charges",


    # =========================================================
    # PRICING & BILLING
    # =========================================================

    "billing dispute or unexpected charge",
    "customer questions monthly service price",
    "unexpected increase in monthly bill",
    "price differs from advertised plan",
    "additional service charge is unclear",
    "installation or activation fee dispute",
    "promotional pricing is unclear",
    "promotional discount missing from bill",
    "customer asks about upgrade pricing",
    "customer asks about downgrade pricing",
    "customer wants cheaper service options",
    "equipment-related charge dispute",
    "customer questions taxes or regulatory fees",
    "customer was charged after cancellation",
    "customer requests package pricing clarification",
    "late payment inquiry",
    "refund inquiry",
    "payment method update failed",
    "payment method change request",
    "customer wants to dispute a previous payment",


    # =========================================================
    # HARDWARE & NETWORK EQUIPMENT
    # =========================================================

    "router or modem not responding",
    "router cannot connect to internet",
    "device repeatedly loses connection",
    "Wi-Fi equipment requires frequent restarting",
    "network device overheating",
    "router lights showing unexpected status",
    "phone cannot connect to router",
    "TV device cannot connect to network",
    "streaming device disconnects repeatedly",
    "customer needs equipment replacement",
    "customer cannot complete device setup",
    "hardware configuration problem",
    "customer needs help resetting equipment",
    "damaged network equipment",
    "device compatibility problem",


    # =========================================================
    # CONTRACT & PLAN MANAGEMENT
    # =========================================================

    "question about contract renewal terms",
    "customer wants to cancel a service or add-on",
    "customer wants to cancel entire service",
    "customer wants to change contract type",
    "customer wants to upgrade service plan",
    "customer wants to downgrade service plan",
    "customer asks about early termination terms",
    "customer asks about contract expiration",
    "customer wants to change an existing add-on",
    "customer asks about available service plans",


    # =========================================================
    # REFERRALS & OFFERS
    # =========================================================

    "referral program question",
    "referral reward missing",
    "referral discount not applied",
    "customer asks about available promotional offers",
    "customer asks about discount eligibility",
    "customer wants to apply an available offer",
    "customer questions promotional offer terms",


    # =========================================================
    # GENERAL SERVICE DISSATISFACTION
    # =========================================================

    "customer unhappy with service reliability",
    "customer expected better service performance",
    "service does not meet advertised expectations",
    "customer reports repeated unresolved problems",
    "customer dissatisfied with support experience",
    "customer has contacted support multiple times",
    "previously resolved problem has returned",
    "customer considers cancelling service",
    "customer requests compensation for poor service",
    "customer requests escalation to specialist",
    "customer reports poor overall service experience",


    # =========================================================
    # SELF-SERVICE & WEBSITE
    # =========================================================

    "customer cannot find required account option",
    "website login not working",
    "password reset fails",
    "customer cannot change service online",
    "billing information unavailable online",
    "service upgrade option missing",
    "service cancellation option difficult to find",
    "online support tools not working",
    "customer cannot view plan details",
    "customer cannot access support history",
    "website form fails during submission",
    "customer unable to update account information",
    "customer cannot find expected self-service option",


    # =========================================================
    # ACCOUNT & CUSTOMER INFORMATION
    # =========================================================

    "customer wants to update personal information",
    "customer wants to update contact information",
    "customer wants to change billing information",
    "customer cannot access account information",
    "customer reports incorrect account details",
    "customer wants to update payment information",


    # =========================================================
    # DATA USAGE & CHARGES
    # =========================================================

    "data usage overage charge",
    "customer questions unexpected data consumption",
    "customer reports unusually high data usage",
    "customer asks how data usage is calculated",
    "customer disputes a data usage charge",


    # =========================================================
    # SERVICE ACTIVATION & INSTALLATION
    # =========================================================

    "new service activation delayed",
    "installation appointment problem",
    "service installation incomplete",
    "new service not working after installation",
    "customer has not received required equipment",
    "customer asks about installation status",
    "activation completed but service unavailable",


    # =========================================================
    # CANCELLATION & RETENTION
    # =========================================================

    "customer wants to cancel service",
    "customer is considering switching providers",
    "customer received a better competitor offer",
    "customer wants a lower monthly price before cancelling",
    "customer is dissatisfied with contract terms",
    "customer wants cancellation fee clarification",
    "customer wants to know retention offers",
]


In [45]:
# customer_chat_topics = [
#     "billing dispute / unexpected charge",
#     "slow internet speed complaint",
#     "service outage in the area",
#     "question about contract renewal terms",
#     "device / router compatibility issue",
#     "request to cancel a service or add-on",
#     "payment method update failed",
#     "streaming service (TV/movies/music) not working",
#     "referral program / discount offer question",
#     "data usage overage charge",
#     "request to upgrade internet plan",
#     "late payment / refund inquiry",
# ]

def pick_topic(row):
    # Bias topic choice slightly using churn risk as a proxy for likely complaint type
    if row['churn_score_target'] >= 70:
        weighted = customer_chat_topics[:8]
    else:
        weighted = customer_chat_topics
    return random.choice(weighted)


def build_chat_prompt(row, topic):
    if row['resolution_status_open_closed'] == 'NotResolved':
        ending = ("End the conversation with the support agent's final line being exactly: "
                   "\"It will be resolved by the end of the day.\"")
    else:
        ending = ("End the conversation with the support agent's final line being exactly: "
                   "\"Thank you for contacting us.\"")

    return f'''Write a realistic customer support chat transcript, 3 to 10 lines total,
alternating "Customer:" and "Agent:" turns.
Topic: {topic}
Customer profile: age {row['age']}, {row['contract_type']} contract, paying ${row['monthly_charges']} per month,
using {row['internet_service']} internet, tenure {row['tenure_months']} months.
{ending}
Only output the chat transcript lines, nothing else (no preamble, no explanation).'''


def build_feedback_prompt(chat_text):
    return f'''Read this customer support chat transcript and write ONE short feedback comment
(1-2 sentences) as if the customer left it afterward, reflecting how they likely feel:

{chat_text}

Feedback:'''


def build_sentiment_prompt(feedback_text):
    return f'''Classify the sentiment of the feedback below as exactly one word:
Positive, Negative, or Neutral. Respond with only that single word.

Feedback: "{feedback_text}"
Sentiment:'''


def clean_sentiment(raw_text):
    t = raw_text.strip().lower()
    if 'pos' in t:
        return 'Positive'
    if 'neg' in t:
        return 'Negative'
    return 'Neutral'


## 11. Generate chats, feedback & sentiment, save `<customer_id>.txt`

Set `TEST_MODE = True` first to validate on a handful of rows before running the full batch.


In [1]:
# ============================================================
# SETTINGS
# ============================================================

TEST_MODE = True
TEST_SIZE = 3

# All customer TXT files will be saved directly here
CUSTOMER_BASE_PATH = "/content/drive/MyDrive/FirstSource/customer_chat_data"

# --- New parallel / batch settings ---
BATCH_SIZE = 25       # how many customers processed per batch
MAX_WORKERS = 8        # concurrent threads per batch (tune based on Ollama server capacity)
SKIP_EXISTING = True    # skip customers whose .txt file already exists (no duplicates)


In [ ]:

import os
import random
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(CUSTOMER_BASE_PATH, exist_ok=True)


# ============================================================
# SELECT DATA
# ============================================================

if TEST_MODE:
    work_df = (
        df.sample(n=TEST_SIZE, random_state=1)
        .reset_index(drop=True)
    )
else:
    work_df = df.reset_index(drop=True)


# ------------------------------------------------------------
# DEDUPLICATION: drop rows whose output file already exists
# ------------------------------------------------------------

if SKIP_EXISTING:
    existing_files = set(os.listdir(CUSTOMER_BASE_PATH))

    def _already_done(customer_id):
        return f"{customer_id}.txt" in existing_files

    before_count = len(work_df)
    work_df = work_df[
        ~work_df["customer_id"].astype(str).apply(_already_done)
    ].reset_index(drop=True)
    skipped_count = before_count - len(work_df)
else:
    skipped_count = 0


print("Customers to process:", len(work_df))
print("Customers skipped (already done):", skipped_count)
print("Output directory:", CUSTOMER_BASE_PATH)


# ============================================================
# WORKER FUNCTION (runs per customer, inside a thread)
# ============================================================

def process_customer(row):
    customer_id = str(row["customer_id"])

    try:
        # ----------------------------------------------------
        # Pick topic
        # ----------------------------------------------------
        topic = pick_topic(row)

        # ----------------------------------------------------
        # Generate chat
        # ----------------------------------------------------
        chat_prompt = build_chat_prompt(row, topic)
        chat_text = ollama_generate(chat_prompt, max_tokens=250)

        # ----------------------------------------------------
        # Generate feedback
        # ----------------------------------------------------
        feedback_prompt = build_feedback_prompt(chat_text)
        feedback_text = ollama_generate(feedback_prompt, max_tokens=80)

        # ----------------------------------------------------
        # Generate sentiment
        # ----------------------------------------------------
        sentiment_prompt = build_sentiment_prompt(feedback_text)
        sentiment_raw = ollama_generate(sentiment_prompt, max_tokens=10)
        sentiment = clean_sentiment(sentiment_raw)

        # ----------------------------------------------------
        # Save customer chat file
        # ----------------------------------------------------
        fname = os.path.join(CUSTOMER_BASE_PATH, f"{customer_id}.txt")

        with open(fname, "w", encoding="utf-8") as f:
            f.write(f"Customer ID: {customer_id}\n")
            f.write(f"Topic: {topic}\n")
            f.write(f"Customer Status: {row['customer_status']}\n")
            f.write("-" * 40 + "\n\n")
            f.write(chat_text)
            f.write("\n\n")
            f.write("-" * 40 + "\n")
            f.write(f"Feedback: {feedback_text}\n")
            f.write(f"Sentiment: {sentiment}\n")

        return {
            "customer_id": customer_id,
            "chat_topic": topic,
            "support_transcript_file": f"{customer_id}.txt",
            "feedback_text": feedback_text,
            "feedback_sentiment": sentiment,
            "status": "ok",
        }

    except Exception as e:
        return {
            "customer_id": customer_id,
            "chat_topic": None,
            "support_transcript_file": None,
            "feedback_text": None,
            "feedback_sentiment": None,
            "status": f"error: {e}",
        }


# ============================================================
# BATCH-WISE PARALLEL EXECUTION
# ============================================================

enrichment_records = []

rows = [row for _, row in work_df.iterrows()]
batches = [rows[i:i + BATCH_SIZE] for i in range(0, len(rows), BATCH_SIZE)]

for batch_idx, batch in enumerate(tqdm(batches, desc="Batches")):

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_customer, row): row for row in batch}

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"Batch {batch_idx + 1}/{len(batches)}",
            leave=False,
        ):
            result = future.result()
            enrichment_records.append(result)


# ============================================================
# CREATE ENRICHMENT DATAFRAME
# ============================================================

enrichment_df = pd.DataFrame(enrichment_records)

# Log any failures separately so you can retry just those
failed_df = enrichment_df[enrichment_df["status"] != "ok"]
if len(failed_df) > 0:
    print(f"\n{len(failed_df)} customers failed — inspect `failed_df` to retry them.")


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\nProcessing completed.")
print("Files saved to:", CUSTOMER_BASE_PATH)
print("\nEnrichment DataFrame:")
display(enrichment_df.head())


## 12. Merge enrichment back into the main table & export
Run cell 11 with `TEST_MODE = False` for the full 2000 rows once you're happy with the sample output.


In [ ]:
final_df = work_df.merge(enrichment_df, on="customer_id", how="left")
final_df.to_csv("telco_churn_enriched.csv", index=False)
print(final_df.shape)
final_df.head()


## 13. Zip transcripts and download everything


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("customer_chats_bundle", "zip", "customer_chats")

files.download("telco_churn_enriched.csv")
files.download("customer_chats_bundle.zip")


## Notes
- Swap `MODEL_NAME` to `gemma2:2b`, `llama3.2:1b`, or any other Ollama model tag for speed/quality trade-offs.
- If Colab disconnects, re-run cells 1-4 (installing Ollama + pulling the model again is required after a runtime reset).
- To scale beyond 2000 rows, just change `N_PER_CLASS` in cell 6.
- `feedback_rating`/`feedback_sentiment` here is a 3-class label (Positive/Negative/Neutral) generated by a second LLM call on top of the first LLM's feedback text, exactly as specified.
